# 04 — Embedding Generation

This notebook implements the fourth stage of the **CLARA** document ingestion pipeline.

It converts the legal chunks produced by `03_document_chunking.ipynb` into dense semantic vector representations using **BGE-M3**, served locally through **Ollama**.

Each embedding remains associated with the metadata of its source chunk. This relationship is essential because the vector will later be stored in Qdrant together with information such as the source document, article, title, page range, and original legal text.

The main outputs are stored under:

```text
data/embeddings/
```

with one embedding file per legal document.

## 1. Imports

This block loads the libraries required for JSON processing, filesystem access, HTTP communication with Ollama, timing, and data inspection.

The embedding stage communicates with the local Ollama service through its HTTP API.

In [1]:
from pathlib import Path
import json
import time

import pandas as pd
import requests


## 2. Project Paths and Configuration

The directories containing processed chunks and generated embeddings are defined relative to the project root.

This block also establishes the configuration required by the embedding pipeline, such as the Ollama endpoint and the embedding model used throughout the corpus.

Using the same embedding model during indexing and query-time retrieval is mandatory because both vectors must belong to the same semantic vector space.

In [2]:
PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "embeddings"

EMBEDDINGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Processed:  {PROCESSED_DIR}")
print(f"Embeddings: {EMBEDDINGS_DIR}")


Processed:  C:\Users\user\Documents\GitHub\clara-rag-pt\data\processed
Embeddings: C:\Users\user\Documents\GitHub\clara-rag-pt\data\embeddings


### Embedding Processing Step 3

This block performs an additional operation in the embedding-generation workflow.

It contributes to transforming the legal retrieval chunks into a validated vector dataset while preserving the metadata required for semantic search and source traceability.

In [3]:
OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "bge-m3"

REQUEST_TIMEOUT = 120

print(f"Ollama: {OLLAMA_URL}")
print(f"Modelo: {EMBEDDING_MODEL}")


Ollama: http://localhost:11434
Modelo: bge-m3


## 2. Project Paths and Configuration

The directories containing processed chunks and generated embeddings are defined relative to the project root.

This block also establishes the configuration required by the embedding pipeline, such as the Ollama endpoint and the embedding model used throughout the corpus.

Using the same embedding model during indexing and query-time retrieval is mandatory because both vectors must belong to the same semantic vector space.

In [4]:
chunks_files = sorted(
    PROCESSED_DIR.glob("*/chunks.json")
)

print(
    f"Foram encontrados {len(chunks_files)} documentos:\n"
)

for path in chunks_files:
    print(f"- {path.parent.name}")


Foram encontrados 10 documentos:

- CIMI
- CIRC
- CIRS
- CIS
- CIVA
- EBF
- LGT
- OE2026
- RGIT
- RITI


## Verificação do Ollama

Antes de gerar milhares de embeddings, verificamos se o serviço Ollama está acessível.


### 4.1 Ollama Availability Check

Before processing the complete corpus, the notebook verifies that the Ollama service is reachable and that the required embedding model is available.

Performing this check early avoids starting a potentially long embedding process with an incorrectly configured local model service.

In [5]:
def check_ollama() -> bool:
    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=10
        )

        response.raise_for_status()

        return True

    except Exception as e:
        print(f"Erro ao contactar o Ollama: {e}")
        return False


ollama_available = check_ollama()

print(
    "Ollama disponível."
    if ollama_available
    else "Ollama indisponível."
)


Ollama disponível.


## 4. Ollama Embedding Interface

This block defines the communication with the local Ollama service used to generate embeddings.

The text of a legal chunk is sent to the configured BGE-M3 model, and the returned numerical vector becomes its semantic representation.

Centralising this operation in a dedicated function makes the embedding process reusable and keeps API handling separate from corpus processing.

In [6]:
def get_installed_models() -> list[str]:
    response = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    return [
        model["name"]
        for model in data.get("models", [])
    ]


if ollama_available:

    installed_models = get_installed_models()

    print("Modelos instalados:")

    for model in installed_models:
        print(f"- {model}")

    model_available = any(
        model == EMBEDDING_MODEL
        or model.startswith(f"{EMBEDDING_MODEL}:")
        for model in installed_models
    )

    if not model_available:
        print(
            f"\nATENÇÃO: {EMBEDDING_MODEL} "
            "não foi encontrado no Ollama."
        )
        print(
            f"Instala primeiro com: "
            f"ollama pull {EMBEDDING_MODEL}"
        )


Modelos instalados:
- bge-m3:latest
- hf.co/ruialexrib/AMALIA-9B-0626-SFT-GGUF:Q3_K_M
- gemma3:4b
- qwen3:4b
- mistral:latest
- qwen2.5:3b
- llama3.1:latest
- llama3:latest


## Função para geração de embeddings

O endpoint `/api/embed` permite enviar texto ao modelo de embeddings e receber o vetor correspondente.


## 4. Ollama Embedding Interface

This block defines the communication with the local Ollama service used to generate embeddings.

The text of a legal chunk is sent to the configured BGE-M3 model, and the returned numerical vector becomes its semantic representation.

Centralising this operation in a dedicated function makes the embedding process reusable and keeps API handling separate from corpus processing.

In [7]:
def generate_embedding(
    text: str,
    model: str = EMBEDDING_MODEL
) -> list[float]:
    """Gera um embedding para um único texto através do Ollama."""

    if not text or not text.strip():
        raise ValueError(
            "Não é possível gerar embedding para texto vazio."
        )

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": model,
            "input": text
        },
        timeout=REQUEST_TIMEOUT
    )

    response.raise_for_status()

    data = response.json()

    embeddings = data.get("embeddings")

    if not embeddings:
        raise ValueError(
            "Ollama não devolveu embeddings."
        )

    return embeddings[0]


## Teste com uma pergunta simples

Este teste permite confirmar o endpoint e determinar a dimensão do vetor produzida pelo modelo.


## 5. Embedding Generation Test

A small embedding request is executed before processing the complete corpus.

This validates the connection to Ollama, confirms that BGE-M3 returns a vector successfully, and allows the embedding dimensionality to be inspected.

In [8]:
test_text = (
    "Tributação dos rendimentos em Portugal."
)

test_embedding = generate_embedding(
    test_text
)

print(
    f"Dimensão do embedding: "
    f"{len(test_embedding)}"
)

print(
    "Primeiros valores:",
    test_embedding[:10]
)


Dimensão do embedding: 1024
Primeiros valores: [-0.019587219, 0.052233983, -0.036666986, -0.019738266, -0.037045013, -0.0038107715, 0.021025378, 0.05670819, 0.01455223, -0.00032917666]


## 5. Embedding Generation Test

A small embedding request is executed before processing the complete corpus.

This validates the connection to Ollama, confirms that BGE-M3 returns a vector successfully, and allows the embedding dimensionality to be inspected.

In [9]:
EMBEDDING_DIMENSION = len(test_embedding)

print(
    f"Dimensão esperada: {EMBEDDING_DIMENSION}"
)


Dimensão esperada: 1024


## Carregar e testar um chunk real


## 5. Embedding Generation Test

A small embedding request is executed before processing the complete corpus.

This validates the connection to Ollama, confirms that BGE-M3 returns a vector successfully, and allows the embedding dimensionality to be inspected.

In [10]:
if not chunks_files:
    raise FileNotFoundError(
        "Não foram encontrados ficheiros chunks.json. "
        "Executa primeiro o Notebook 3."
    )

test_chunks_path = chunks_files[0]

with open(
    test_chunks_path,
    "r",
    encoding="utf-8"
) as f:
    test_document = json.load(f)

test_chunk = test_document["chunks"][0]

print(f"Documento: {test_chunk['document_id']}")
print(f"Chunk ID: {test_chunk['chunk_id']}")
print(f"Artigo: {test_chunk['article']}")
print()
print(test_chunk["embedding_text"][:1500])


Documento: CIMI
Chunk ID: CIMI_ART_0001_CHUNK_001
Artigo: Artigo 13.º

CIMI
CAPÍTULO III
Artigo 13.º
Elementos para avaliações

1 - Os elementos referidos nas alíneas a) a d) do n.º 1 do artigo 62.º do CIMI deverão ser aprovados no prazo de 180 dias a contar da data da publicação do presente diploma. 2 - As plantas previstas na alínea b) do artigo 128.º do CIMI devem ser remetidas aos serviços de finanças da área do município no prazo de 30 dias a contar da data da publicação do presente diploma. 3 - As câmaras municipais devem colaborar com os serviços competentes da Direcção-Geral dos Impostos na elaboração das propostas de fixação dos elementos de avaliação, as quais devem ser-lhes previamente remetidas para que se pronunciem no prazo de 15 dias.


## 5. Embedding Generation Test

A small embedding request is executed before processing the complete corpus.

This validates the connection to Ollama, confirms that BGE-M3 returns a vector successfully, and allows the embedding dimensionality to be inspected.

In [11]:
chunk_embedding = generate_embedding(
    test_chunk["embedding_text"]
)

print(
    f"Dimensão: {len(chunk_embedding)}"
)

assert (
    len(chunk_embedding)
    ==
    EMBEDDING_DIMENSION
), "Dimensão inesperada do embedding."

print("Embedding gerado corretamente.")


Dimensão: 1024
Embedding gerado corretamente.


## Geração dos embeddings por documento

Cada registo contém o vetor e os metadados necessários para a indexação posterior no Qdrant.


### 5.1 Embedding Dimension Validation

The dimensionality of the generated vector is inspected.

All vectors stored in the same Qdrant collection must have identical dimensions. This validation therefore confirms an important structural requirement before corpus-wide generation begins.

In [12]:
def embed_document(
    chunked_document: dict,
    model: str = EMBEDDING_MODEL,
    show_progress: bool = True
) -> dict:
    """Gera embeddings para todos os chunks de um documento."""

    document_id = chunked_document["document_id"]
    chunks = chunked_document["chunks"]

    embedded_chunks = []

    start_time = time.time()

    for index, chunk in enumerate(
        chunks,
        start=1
    ):

        embedding = generate_embedding(
            chunk["embedding_text"],
            model=model
        )

        if (
            len(embedding)
            != EMBEDDING_DIMENSION
        ):
            raise ValueError(
                f"Dimensão inesperada no chunk "
                f"{chunk['chunk_id']}: "
                f"{len(embedding)}"
            )

        embedded_chunks.append({
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "article_index": chunk.get("article_index"),
            "article": chunk["article"],
            "article_number": chunk["article_number"],
            "article_title": chunk.get("article_title"),
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "hierarchy": chunk.get("hierarchy", {}),
            "chunk_index": chunk["chunk_index"],
            "total_chunks": chunk["total_chunks"],
            "word_count": chunk["word_count"],
            "text": chunk["text"],
            "embedding_text": chunk["embedding_text"],
            "embedding": embedding
        })

        if (
            show_progress
            and (
                index == 1
                or index % 25 == 0
                or index == len(chunks)
            )
        ):
            print(
                f"{document_id}: "
                f"{index}/{len(chunks)}"
            )

    elapsed = time.time() - start_time

    return {
        "document_id": document_id,
        "source_file": chunked_document["source_file"],
        "embedding_model": model,
        "embedding_dimension": EMBEDDING_DIMENSION,
        "num_chunks": len(embedded_chunks),
        "elapsed_seconds": round(elapsed, 2),
        "chunks": embedded_chunks
    }


## 7. Saving Embedding Records

Generated embeddings are serialised to JSON together with the metadata required for later vector indexing.

The output is organised by legal document under:

```text
data/embeddings/<DOCUMENT_ID>_embeddings.json
```

This intermediate representation separates model inference from Qdrant indexing, allowing either stage to be rerun independently.

In [13]:
def save_embeddings(
    embedded_document: dict,
    output_dir: Path
) -> Path:
    """Guarda os embeddings de um documento em JSON."""

    document_id = embedded_document[
        "document_id"
    ]

    output_path = (
        output_dir
        / f"{document_id}_embeddings.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            embedded_document,
            f,
            ensure_ascii=False
        )

    return output_path


## Processar todos os documentos

Esta célula pode demorar vários minutos, dependendo do número de chunks e do hardware disponível.


## 2. Project Paths and Configuration

The directories containing processed chunks and generated embeddings are defined relative to the project root.

This block also establishes the configuration required by the embedding pipeline, such as the Ollama endpoint and the embedding model used throughout the corpus.

Using the same embedding model during indexing and query-time retrieval is mandatory because both vectors must belong to the same semantic vector space.

In [14]:
results = []

for chunks_path in chunks_files:

    document_id = chunks_path.parent.name

    print(
        f"\nA processar {document_id}..."
    )

    try:

        with open(
            chunks_path,
            "r",
            encoding="utf-8"
        ) as f:
            chunked_document = json.load(f)

        embedded_document = embed_document(
            chunked_document,
            model=EMBEDDING_MODEL,
            show_progress=True
        )

        output_path = save_embeddings(
            embedded_document,
            EMBEDDINGS_DIR
        )

        results.append({
            "document_id": document_id,
            "chunks": embedded_document[
                "num_chunks"
            ],
            "dimension": embedded_document[
                "embedding_dimension"
            ],
            "seconds": embedded_document[
                "elapsed_seconds"
            ],
            "output": str(output_path),
            "status": "OK"
        })

        print(
            f"{document_id}: OK "
            f"({embedded_document['num_chunks']} embeddings)"
        )

    except Exception as e:

        results.append({
            "document_id": document_id,
            "chunks": None,
            "dimension": None,
            "seconds": None,
            "output": None,
            "status": f"ERROR: {e}"
        })

        print(
            f"{document_id}: ERRO — {e}"
        )



A processar CIMI...
CIMI: 1/226
CIMI: 25/226
CIMI: 50/226
CIMI: 75/226
CIMI: 100/226
CIMI: 125/226
CIMI: 150/226
CIMI: 175/226
CIMI: 200/226
CIMI: 225/226
CIMI: 226/226
CIMI: OK (226 embeddings)

A processar CIRC...
CIRC: 1/305
CIRC: 25/305
CIRC: 50/305
CIRC: 75/305
CIRC: 100/305
CIRC: 125/305
CIRC: 150/305
CIRC: 175/305
CIRC: 200/305
CIRC: 225/305
CIRC: 250/305
CIRC: 275/305
CIRC: 300/305
CIRC: 305/305
CIRC: OK (305 embeddings)

A processar CIRS...
CIRS: 1/370
CIRS: 25/370
CIRS: 50/370
CIRS: 75/370
CIRS: 100/370
CIRS: 125/370
CIRS: 150/370
CIRS: 175/370
CIRS: 200/370
CIRS: 225/370
CIRS: 250/370
CIRS: 275/370
CIRS: 300/370
CIRS: 325/370
CIRS: 350/370
CIRS: 370/370
CIRS: OK (370 embeddings)

A processar CIS...
CIS: 1/130
CIS: 25/130
CIS: 50/130
CIS: 75/130
CIS: 100/130
CIS: 125/130
CIS: 130/130
CIS: OK (130 embeddings)

A processar CIVA...
CIVA: 1/354
CIVA: 25/354
CIVA: 50/354
CIVA: 75/354
CIVA: 100/354
CIVA: 125/354
CIVA: 150/354
CIVA: 175/354
CIVA: 200/354
CIVA: 225/354
CIVA: 250/354

## 9. Embedding Output Inspection

The generated embedding records are represented in tabular form for validation.

This makes it easier to confirm that document identifiers, article metadata, chunk identifiers, and vector information remain correctly associated after model inference.

In [15]:
df_results = pd.DataFrame(results)

df_results


,document_id,chunks,dimension,seconds,output,status
0,CIMI,226,1024,95.17,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
1,CIRC,305,1024,125.32,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
2,CIRS,370,1024,143.72,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
3,CIS,130,1024,51.02,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
4,CIVA,354,1024,154.55,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
5,EBF,279,1024,113.02,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
6,LGT,155,1024,63.72,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
7,OE2026,404,1024,169.89,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
8,RGIT,165,1024,67.71,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK
9,RITI,168,1024,65.69,C:\Users\user\Documents\GitHub\clara-rag-pt\da...,OK


## Validação global dos embeddings


## 2. Project Paths and Configuration

The directories containing processed chunks and generated embeddings are defined relative to the project root.

This block also establishes the configuration required by the embedding pipeline, such as the Ollama endpoint and the embedding model used throughout the corpus.

Using the same embedding model during indexing and query-time retrieval is mandatory because both vectors must belong to the same semantic vector space.

In [16]:
embedding_files = sorted(
    EMBEDDINGS_DIR.glob("*_embeddings.json")
)

validation = []

all_chunk_ids = []

for embedding_path in embedding_files:

    with open(
        embedding_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    dimensions = {
        len(chunk["embedding"])
        for chunk in data["chunks"]
    }

    chunk_ids = [
        chunk["chunk_id"]
        for chunk in data["chunks"]
    ]

    all_chunk_ids.extend(chunk_ids)

    validation.append({
        "document_id": data["document_id"],
        "chunks": len(data["chunks"]),
        "dimensions": sorted(dimensions),
        "empty_embeddings": sum(
            1
            for chunk in data["chunks"]
            if not chunk["embedding"]
        )
    })

df_validation = pd.DataFrame(validation)

df_validation


,document_id,chunks,dimensions,empty_embeddings
0,CIMI,226,[1024],0
1,CIRC,305,[1024],0
2,CIRS,370,[1024],0
3,CIS,130,[1024],0
4,CIVA,354,[1024],0
5,EBF,279,[1024],0
6,LGT,155,[1024],0
7,OE2026,404,[1024],0
8,RGIT,165,[1024],0
9,RITI,168,[1024],0


### 5.1 Embedding Dimension Validation

The dimensionality of the generated vector is inspected.

All vectors stored in the same Qdrant collection must have identical dimensions. This validation therefore confirms an important structural requirement before corpus-wide generation begins.

In [17]:
print(
    f"Total de embeddings: {len(all_chunk_ids)}"
)

print(
    f"Chunk IDs únicos: {len(set(all_chunk_ids))}"
)

assert (
    len(all_chunk_ids)
    ==
    len(set(all_chunk_ids))
), "Existem chunk_id duplicados."

assert all(
    row["dimensions"]
    == [EMBEDDING_DIMENSION]
    for row in validation
), "Foram encontradas dimensões inconsistentes."

assert all(
    row["empty_embeddings"] == 0
    for row in validation
), "Foram encontrados embeddings vazios."

print(
    "Validação concluída: "
    "embeddings consistentes e IDs únicos."
)


Total de embeddings: 2556
Chunk IDs únicos: 2556
Validação concluída: embeddings consistentes e IDs únicos.


## Estatísticas finais


### 5.1 Embedding Dimension Validation

The dimensionality of the generated vector is inspected.

All vectors stored in the same Qdrant collection must have identical dimensions. This validation therefore confirms an important structural requirement before corpus-wide generation begins.

In [18]:
total_embeddings = sum(
    item["chunks"]
    for item in validation
)

print(
    f"Documentos: {len(validation)}"
)

print(
    f"Total de embeddings: "
    f"{total_embeddings}"
)

print(
    f"Dimensão dos vetores: "
    f"{EMBEDDING_DIMENSION}"
)

print(
    f"Modelo: {EMBEDDING_MODEL}"
)


Documentos: 10
Total de embeddings: 2556
Dimensão dos vetores: 1024
Modelo: bge-m3


## Next Step

The vector representations generated by this notebook are the input to:

```text
05_qdrant_indexing.ipynb
```

The next notebook creates the CLARA vector collection in **Qdrant** and uploads each embedding together with its legal metadata.

After indexing, the corpus becomes available for semantic similarity search using natural-language questions.
